# MAPPO (EPyMARL, PettingZoo MPE) - Colab Notebook

This notebook runs **MAPPO** using EPyMARL's built-in implementation on `simple_spread` and reproduces the same 3x2 training-chart layout used in `ippo/ippo_colab.ipynb`.

1. Set runtime to GPU (optional).
2. Run cells top-to-bottom.
3. Update constants in one place for future sweeps.
            


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/uoe-agents/epymarl.git"
REPO_DIR = Path("epymarl")

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pettingzoo[mpe]", "supersuit", "matplotlib", "numpy"])
print("Dependencies ready.")


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# =========================
# Experiment constants
# =========================
ALGO_CONFIG = "mappo"
EXPERIMENT_NAME = "mappo_simple_spread_colab"

ENV_KEY = "pz-mpe-simple-spread-v3"
NUM_AGENTS = 5
MAX_CYCLES = 100
LOCAL_RATIO = 0.5
CONTINUOUS_ACTIONS = False

TOTAL_TIMESTEPS = 200_000
ROLLOUT_LENGTH = 2048  # retained to mirror ippo_colab constants (EPyMARL uses episode batches)

SEED = 42
USE_CUDA = False

TEST_INTERVAL = 10_000
LOG_INTERVAL = 10_000
TEST_EPISODES = 10

RESULTS_ROOT = Path("epymarl") / "results" / "sacred"
PLOT_PATH = "mappo_training_colab.png"

# Sacred metric key preference order
POLICY_LOSS_KEYS = ["actor_loss", "policy_loss", "loss"]
VALUE_LOSS_KEYS = ["critic_loss", "value_loss", "loss"]
ENTROPY_KEYS = ["entropy_loss", "entropy"]
BELLMAN_KEYS = ["critic_loss", "td_error_abs", "loss"]  # MAPPO proxy: critic regression loss
RETURN_KEYS = ["return_mean", "test_return_mean"]
EP_LENGTH_KEYS = ["ep_length_mean", "episode_length_mean", "episode_limit_mean"]

ENTROPY_FALLBACK_VALUE = np.nan
BELLMAN_FALLBACK_TO_VALUE_LOSS = True


`mean_bellman_error` is mapped from MAPPO critic-loss-style metrics because EPyMARL MAPPO does not log a separate TD-error scalar in the same form as your IPPO implementation.
            


In [ ]:
def _numeric_run_dirs(root: Path):
    if not root.exists():
        return []
    return sorted([p for p in root.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name))


def run_epymarl_experiment():
    before = {p.name for p in _numeric_run_dirs(RESULTS_ROOT)}

    cmd = [
        sys.executable,
        "src/main.py",
        f"--config={ALGO_CONFIG}",
        "--env-config=gymma",
        "with",
        f"name={EXPERIMENT_NAME}",
        f"seed={SEED}",
        f"use_cuda={USE_CUDA}",
        f"t_max={TOTAL_TIMESTEPS}",
        f"test_interval={TEST_INTERVAL}",
        f"log_interval={LOG_INTERVAL}",
        f"runner_log_interval={LOG_INTERVAL}",
        f"test_nepisode={TEST_EPISODES}",
        f"env_args.key={ENV_KEY}",
        f"env_args.time_limit={MAX_CYCLES}",
        f"env_args.N={NUM_AGENTS}",
        f"env_args.local_ratio={LOCAL_RATIO}",
        f"env_args.continuous_actions={CONTINUOUS_ACTIONS}",
    ]

    proc = subprocess.Popen(
        cmd,
        cwd="epymarl",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")

    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"EPyMARL run failed with exit code {rc}")

    after_dirs = _numeric_run_dirs(RESULTS_ROOT)
    created = [p for p in after_dirs if p.name not in before]
    run_dir = created[-1] if created else (after_dirs[-1] if after_dirs else None)
    if run_dir is None:
        raise FileNotFoundError("No Sacred run directory found under epymarl/results/sacred")
    return run_dir


def load_sacred_metrics(run_dir: Path):
    metrics_path = run_dir / "metrics.json"
    if not metrics_path.exists():
        raise FileNotFoundError(f"Missing metrics file: {metrics_path}")
    with metrics_path.open("r") as f:
        return json.load(f)


def pick_series(metrics, keys):
    for key in keys:
        if key in metrics and metrics[key].get("values"):
            vals = np.asarray(metrics[key]["values"], dtype=np.float32)
            return vals, key
    return np.asarray([], dtype=np.float32), None


def pad_to_length(arr, length, fill=np.nan):
    out = np.full(length, fill, dtype=np.float32)
    if len(arr) > 0:
        out[: min(len(arr), length)] = arr[:length]
    return out


def build_metrics_history(metrics):
    policy_loss, policy_key = pick_series(metrics, POLICY_LOSS_KEYS)
    value_loss, value_key = pick_series(metrics, VALUE_LOSS_KEYS)
    entropy, entropy_key = pick_series(metrics, ENTROPY_KEYS)
    bellman, bellman_key = pick_series(metrics, BELLMAN_KEYS)
    episode_return, return_key = pick_series(metrics, RETURN_KEYS)
    episode_length, ep_length_key = pick_series(metrics, EP_LENGTH_KEYS)

    train_len = max(len(policy_loss), len(value_loss), len(entropy), len(bellman), 1)
    policy_loss = pad_to_length(policy_loss, train_len)
    value_loss = pad_to_length(value_loss, train_len)

    if len(entropy) == 0:
        entropy = np.full(train_len, ENTROPY_FALLBACK_VALUE, dtype=np.float32)
    else:
        entropy = pad_to_length(entropy, train_len)

    if len(bellman) == 0 and BELLMAN_FALLBACK_TO_VALUE_LOSS:
        bellman = value_loss.copy()
    else:
        bellman = pad_to_length(bellman, train_len)

    if len(episode_length) == 0:
        episode_length = np.full(len(episode_return), float(MAX_CYCLES), dtype=np.float32)

    ep_len = max(len(episode_return), 1)
    episode_return = pad_to_length(episode_return, ep_len)
    episode_length = pad_to_length(episode_length, ep_len, fill=float(MAX_CYCLES))
    episode_reward = episode_return / np.maximum(episode_length, 1e-8)

    history = {
        "iterations": list(range(1, train_len + 1)),
        "episode_iterations": list(range(1, ep_len + 1)),
        "policy_loss": policy_loss,
        "value_loss": value_loss,
        "entropy": entropy,
        "mean_bellman_error": bellman,
        "mean_episode_return": episode_return,
        "mean_episode_reward": episode_reward,
    }

    chosen_keys = {
        "policy_loss": policy_key,
        "value_loss": value_key,
        "entropy": entropy_key,
        "mean_bellman_error": bellman_key,
        "mean_episode_return": return_key,
        "mean_episode_length": ep_length_key,
    }
    return history, chosen_keys


def plot_metrics(metrics_history, save_path):
    plt.style.use("ggplot")
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))

    iteration_x = metrics_history["iterations"]
    episode_iteration_x = metrics_history["episode_iterations"]

    axes[0, 0].plot(iteration_x, metrics_history["policy_loss"])
    axes[0, 0].set_title("policy_loss")
    axes[0, 0].set_xlabel("Iteration")
    axes[0, 0].set_ylabel("policy_loss")
    axes[0, 0].grid(True, alpha=0.4)

    axes[0, 1].plot(iteration_x, metrics_history["value_loss"])
    axes[0, 1].set_title("value_loss")
    axes[0, 1].set_xlabel("Iteration")
    axes[0, 1].set_ylabel("value_loss")
    axes[0, 1].grid(True, alpha=0.4)

    axes[1, 0].plot(iteration_x, metrics_history["entropy"])
    axes[1, 0].set_title("entropy")
    axes[1, 0].set_xlabel("Iteration")
    axes[1, 0].set_ylabel("entropy")
    axes[1, 0].grid(True, alpha=0.4)

    axes[1, 1].plot(iteration_x, metrics_history["mean_bellman_error"])
    axes[1, 1].set_title("mean_bellman_error")
    axes[1, 1].set_xlabel("Iteration")
    axes[1, 1].set_ylabel("mean_bellman_error")
    axes[1, 1].grid(True, alpha=0.4)

    axes[2, 0].plot(episode_iteration_x, metrics_history["mean_episode_return"])
    axes[2, 0].set_title("mean_episode_return")
    axes[2, 0].set_xlabel("Iteration")
    axes[2, 0].set_ylabel("mean_episode_return")
    axes[2, 0].grid(True, alpha=0.4)

    axes[2, 1].plot(episode_iteration_x, metrics_history["mean_episode_reward"])
    axes[2, 1].set_title("mean_episode_rewards")
    axes[2, 1].set_xlabel("Iteration")
    axes[2, 1].set_ylabel("mean_episode_rewards")
    axes[2, 1].grid(True, alpha=0.4)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Metrics plot saved to {save_path}")


In [ ]:
run_dir = run_epymarl_experiment()
metrics = load_sacred_metrics(run_dir)
metrics_history, chosen_keys = build_metrics_history(metrics)

print("Sacred run:", run_dir)
print("Metric key mapping:", chosen_keys)
plot_metrics(metrics_history, PLOT_PATH)
